# 04_als_model

In [1]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession

# 1. Point to your Hadoop bin folder
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

# 2. Pin Python
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 3. Start Spark with the RawLocalFileSystem workaround
spark = SparkSession.builder \
    .appName("Spotify_ALS") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.home.dir", r"C:\hadoop") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .getOrCreate()

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Define the path to your split data
SPLIT_DIR = "../outputs/outputs/train_test_split"

# Load the train and test sets (Assuming they were saved as parquet or csv)
# Note: If your teammate saved them as CSV, change .parquet to .csv(header=True, inferSchema=True)
users_train = spark.read.csv(f"{SPLIT_DIR}/users_train", header=True, inferSchema=True)
users_test = spark.read.csv(f"{SPLIT_DIR}/users_test", header=True, inferSchema=True)

print(f"Training rows: {users_train.count()}")
users_train.show(5)

Training rows: 9107237
+-------+-------+-----------+--------------------+--------------------+-------------------+
|user_id|song_id|interaction|    user_id_original|          track_name|        artist_name|
+-------+-------+-----------+--------------------+--------------------+-------------------+
|      0| 196963|        1.0|00055176fea33f6e0...|Another You (Anot...|Against The Current|
|      0| 920045|        1.0|00055176fea33f6e0...|       Gone Too Soon|        Simple Plan|
|      0|1143595|        1.0|00055176fea33f6e0...|   If You Don't Know|5 Seconds Of Summer|
|      0|1730322|        1.0|00055176fea33f6e0...| One Of Those Nights|       Shawn Mendes|
|      0|1946760|        1.0|00055176fea33f6e0...|         Risk It All|          The Vamps|
+-------+-------+-----------+--------------------+--------------------+-------------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.recommendation import ALS

# Initialize the ALS model using the columns your teammate already prepared
als = ALS(
    maxIter=10,                      
    regParam=0.1,                    
    userCol="user_id",         # Already an integer thanks to Notebook 02.2
    itemCol="song_id",         # Already an integer thanks to Notebook 02.2
    ratingCol="interaction",   # Your teammate renamed 'play_count' to 'interaction'
    coldStartStrategy="drop",        
    implicitPrefs=True               
)

print("Training the ALS Model... ")
model = als.fit(users_train)
print("Training Complete!")

Training the ALS Model... (This might take a minute or two)
Training Complete!
